In [1]:
from google.colab import drive

# Autoriser l'accès à Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import pandas as pd, io, re
from pathlib import Path

chemin = Path('/content/drive/MyDrive/Manal El ghafiri- Soha El ghamri Fin 2 Tp2/FMI.xls')
if not chemin.is_file():
    raise FileNotFoundError(f'Fichier introuvable : {chemin}')

# 1) lire le texte (essayer plusieurs encodages)
texte = None
for enc in ('utf-8-sig', 'utf-16', 'cp1252', 'latin-1'):
    try:
        texte = chemin.read_text(encoding=enc)
        break
    except UnicodeError:
        continue

lignes = [l for l in texte.splitlines() if l.strip()]

# 2) trouver la ligne d'en-tête (celle qui commence par "Date")
i_entete = next(i for i, l in enumerate(lignes) if l.strip().startswith('Date'))

# 3) garder seulement les lignes de données ("Mois jj, aaaa")
motif = re.compile(r'^\s*[A-Z][a-z]+ \d{1,2}, \d{4}\t')
donnees = [l for l in lignes[i_entete + 1:] if motif.match(l)]

df = pd.read_csv(io.StringIO('\n'.join([lignes[i_entete]] + donnees)),
                 sep='\t', thousands=',')
df.columns = ['date', 'titres', 'compte_1', 'compte_2', 'cva', 'cva_2', 'total']

df['date'] = pd.to_datetime(df['date'].str.strip(), format='%B %d, %Y')
df['annee'] = df['date'].dt.year

display(df.head(10))
print('Dimensions :', df.shape)

,date,titres,compte_1,compte_2,cva,cva_2,total,annee
0,2025-09-30,767434350,19929540,475158,-43048809,0,744790239,2025
1,2024-12-31,724385541,557629540,475158,0,0,1282490239,2024
2,2023-12-31,724385541,1519729540,876477,0,0,2244991558,2023
3,2022-12-31,724385541,1522462451,63035,0,0,2246911027,2022
4,2021-12-31,724385541,1522462451,67725,0,0,2246915717,2021
5,2020-12-31,724385541,2173462451,72644,0,1,2897920637,2020
6,2019-12-31,732687201,14360791,74704,0,0,747122696,2019
7,2018-12-31,732687201,14360791,79250,0,0,747127242,2018
8,2017-12-31,732687201,14692637,2943,0,0,747382781,2017
9,2016-12-31,732687201,14700011,8806,0,0,747396018,2016


Dimensions : (42, 8)


In [16]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
df.to_sql('avoirs_fmi', conn, index=False, if_exists='replace')

def executer_sql(titre, requete):
    print(f"=== {titre} ===")
    display(pd.read_sql_query(requete, conn))
    print("\n")

# 1. Toutes les observations
executer_sql("1. Toutes les observations", "SELECT * FROM avoirs_fmi;")

# 2. Choix de colonnes + calcul : part des titres dans le total
executer_sql("2. Part des titres dans le total", """
SELECT annee, titres, total,
       ROUND(titres * 100.0 / total, 1) AS part_titres_pct
FROM avoirs_fmi;
""")

# 3. LIMIT : les 3 années les plus récentes
executer_sql("3. Les 3 années les plus récentes", """
SELECT annee, total FROM avoirs_fmi
ORDER BY annee DESC
LIMIT 3;
""")

# 4. ORDER BY : classement selon le compte n°1
executer_sql("4. Tri selon le compte n°1 (décroissant)", """
SELECT annee, compte_1
FROM avoirs_fmi
ORDER BY compte_1 DESC;
""")

# 5. WHERE : années où le total est inférieur à 600 millions
executer_sql("5. Années où le total est inférieur à 600 millions de DTS", """
SELECT annee, total FROM avoirs_fmi
WHERE total < 600000000
ORDER BY annee;
""")

# 6. COUNT : nombre d'années où le total dépasse 1 milliard
executer_sql("6. Nombre d'années avec un total > 1 milliard de DTS", """
SELECT COUNT(*) AS nb_annees FROM avoirs_fmi
WHERE total > 1000000000;
""")

# 7. AVG : moyenne des titres depuis 2010
executer_sql("7. Moyenne des titres depuis 2010", """
SELECT ROUND(AVG(titres)) AS moyenne_titres
FROM avoirs_fmi
WHERE annee >= 2010;
""")

# 8. MIN / MAX des titres
executer_sql("8. Minimum et maximum des titres", """
SELECT MIN(titres) AS min_titres, MAX(titres) AS max_titres
FROM avoirs_fmi;
""")

# 9. GROUP BY : périodes de 5 ans
executer_sql("9. Moyenne du total par période de 5 ans", """
SELECT (annee/5)*5 AS debut_periode,
       COUNT(*) AS nb_annees,
       ROUND(AVG(total)) AS total_moyen
FROM avoirs_fmi
GROUP BY debut_periode
ORDER BY debut_periode;
""")

# 10. HAVING : périodes où le compte n°1 a dépassé 1 milliard
executer_sql("10. Périodes de 5 ans où le compte n°1 a dépassé 1 milliard", """
SELECT (annee/5)*5 AS debut_periode,
       MAX(compte_1) AS max_compte_1
FROM avoirs_fmi
GROUP BY debut_periode
HAVING MAX(compte_1) > 1000000000
ORDER BY debut_periode;
""")

# 11. AND / OR
executer_sql("11. AND : depuis 2000 et total inférieur à 600 millions", """
SELECT annee, total FROM avoirs_fmi
WHERE annee >= 2000 AND total < 600000000
ORDER BY annee;
""")

executer_sql("11. OR : total > 2 milliards OU compte CVA négatif", """
SELECT annee, cva, total FROM avoirs_fmi
WHERE total > 2000000000 OR cva < 0
ORDER BY annee;
""")

# 12. BETWEEN et IN
executer_sql("12. BETWEEN : période 2015-2019", """
SELECT annee, total FROM avoirs_fmi
WHERE annee BETWEEN 2015 AND 2019
ORDER BY annee;
""")

executer_sql("12. IN : années choisies", """
SELECT annee, total FROM avoirs_fmi
WHERE annee IN (1987, 1998, 2008, 2020, 2025)
ORDER BY annee;
""")

# 13. RANK : classement selon le compte n°1
executer_sql("13. Classement des 8 plus hauts niveaux du compte n°1", """
SELECT annee, compte_1,
       RANK() OVER (ORDER BY compte_1 DESC) AS rang
FROM avoirs_fmi
ORDER BY rang
LIMIT 8;
""")

# 14. Sous-requête : années sous la moyenne
executer_sql("14. Années où le total est sous la moyenne", """
SELECT annee, total
FROM avoirs_fmi
WHERE total < (SELECT AVG(total) FROM avoirs_fmi)
ORDER BY annee;
""")

# 15. Synthèse : tendance annuelle (hausse / baisse)
executer_sql("15. Synthèse : tendance du total d'une année à l'autre", """
SELECT annee, total, variation,
  CASE WHEN variation IS NULL THEN 'Départ'
       WHEN variation > 0 THEN 'Hausse'
       WHEN variation < 0 THEN 'Baisse'
       ELSE 'Stable' END AS tendance
FROM (
    SELECT annee, total,
           total - LAG(total) OVER (ORDER BY annee) AS variation
    FROM avoirs_fmi
)
ORDER BY annee DESC;
""")

=== 1. Toutes les observations ===


,date,titres,compte_1,compte_2,cva,cva_2,total,annee
0,2025-09-30 00:00:00,767434350,19929540,475158,-43048809,0,744790239,2025
1,2024-12-31 00:00:00,724385541,557629540,475158,0,0,1282490239,2024
2,2023-12-31 00:00:00,724385541,1519729540,876477,0,0,2244991558,2023
3,2022-12-31 00:00:00,724385541,1522462451,63035,0,0,2246911027,2022
4,2021-12-31 00:00:00,724385541,1522462451,67725,0,0,2246915717,2021
5,2020-12-31 00:00:00,724385541,2173462451,72644,0,1,2897920637,2020
6,2019-12-31 00:00:00,732687201,14360791,74704,0,0,747122696,2019
7,2018-12-31 00:00:00,732687201,14360791,79250,0,0,747127242,2018
8,2017-12-31 00:00:00,732687201,14692637,2943,0,0,747382781,2017
9,2016-12-31 00:00:00,732687201,14700011,8806,0,0,747396018,2016




=== 2. Part des titres dans le total ===


,annee,titres,total,part_titres_pct
0,2025,767434350,744790239,103.0
1,2024,724385541,1282490239,56.5
2,2023,724385541,2244991558,32.3
3,2022,724385541,2246911027,32.2
4,2021,724385541,2246915717,32.2
5,2020,724385541,2897920637,25.0
6,2019,732687201,747122696,98.1
7,2018,732687201,747127242,98.1
8,2017,732687201,747382781,98.0
9,2016,732687201,747396018,98.0




=== 3. Les 3 années les plus récentes ===


...

,annee,compte_1,rang
0,2020,2173462451,1
1,2022,1522462451,2
2,2021,1522462451,2
3,2023,1519729540,4
4,2024,557629540,5
5,1984,60731903,6
6,1985,26936217,7
7,2025,19929540,8




=== 14. Années où le total est sous la moyenne ===


,annee,total
0,1990,833065016
1,1991,708111375
2,1992,716493364
3,1993,604534837
4,1994,498451252
5,1995,432218633
6,1996,399686886
7,1997,397386938
8,1998,397386805
9,1999,517761435




=== 15. Synthèse : tendance du total d'une année à l'autre ===


,annee,total,variation,tendance
0,2025,744790239,-5.377000e+08,Baisse
1,2024,1282490239,-9.625013e+08,Baisse
2,2023,2244991558,-1.919469e+06,Baisse
3,2022,2246911027,-4.690000e+03,Baisse
4,2021,2246915717,-6.510049e+08,Baisse
5,2020,2897920637,2.150798e+09,Hausse
6,2019,747122696,-4.546000e+03,Baisse
7,2018,747127242,-2.555390e+05,Baisse
8,2017,747382781,-1.323700e+04,Baisse
9,2016,747396018,2.296500e+08,Hausse
